# 10장 코딩 에이전트

코드 작성에 사용할 에이전트를 구현합니다.

[Maarten Grootendorst](https://www.linkedin.com/in/mgrootendorst/)와 [Jay Alammar](https://www.linkedin.com/in/jalammar)의 도서 [An Illustrated Guide to AI Agents](https://www.oreilly.com/library/view/an-illustrated-guide/9798341662681/)에 수록된 10장 실습입니다. [전체 실습 목차](../README.md) · [원본 저장소](https://github.com/HandsOnLLM/An-Illustrated-Guide-To-AI-Agents)

### Google Colab에서 패키지 설치하기 (선택)

Colab 등 클라우드 노트북에서는 아래 설치 셀의 주석을 해제하고 실행합니다. 원격 모델 API를 사용한다면 Ollama 설치 셀은 건너뜁니다.

In [ ]:
# %%capture
# !pip install illustrated-agents

Colab 안에서 Ollama로 모델을 실행하려면 **런타임 → 런타임 유형 변경 → 하드웨어 가속기 → T4 GPU**를 선택한 뒤, 아래 셀의 주석을 해제하고 실행합니다.

In [ ]:
# !apt-get install -y zstd > /dev/null 2>&1 && curl -fsSL https://ollama.com/install.sh | sh
# !nohup ollama serve > /dev/null 2>&1 & sleep 3 && ollama pull gemma4:e4b &

---

## 1 - 언어 모델 선택

이번 실습에 사용할 언어 모델을 지정합니다.

In [ ]:
from illustrated_agents.chapters.ch2 import LLM

# Gemma 4 E4B(내장 추론과 도구 호출 지원)
llm = LLM(model="gemma4:e4b", think=True)

다른 언어 모델을 사용하려면 아래에서 로컬 서버 또는 클라우드 API에 해당하는 설정을 선택합니다.

In [ ]:
# # llama.cpp
# llm = LLM(model="gemma-4-e4B-it-Q4_K_M", base_url="http://127.0.0.1:8080")

# # LM Studio
# llm = LLM(model="gemma-4-e4b-it", base_url="http://127.0.0.1:1234/v1")

# # OpenRouter
# import os
# llm = LLM(model="google/gemma-4-31b-it", base_url="https://openrouter.ai/api/v1", api_key=os.environ["OPENROUTER_API_KEY"])

## 2. 코딩 도구

코딩 도구는 언어 모델을 소프트웨어 환경과 연결합니다. 파일 도구로는 목록을 조회하고 내용을 읽거나 쓸 수 있습니다. 코드 인터프리터는 모델이 작성한 프로그램을 실행합니다.

`read_file`과 `list_files`는 기존 파일을 살펴보고, `write_file`은 파일을 만들거나 수정합니다. `execute_python`은 제한 시간이 있는 하위 프로세스에서 파이썬 코드를 실행합니다. `show_python`은 코드에 서식을 적용해 화면에 표시합니다.

In [ ]:
import subprocess
import sys
from pathlib import Path


def read_file(path: str) -> str:
    """Read a file's contents."""
    target = Path(path)
    if not target.exists():
        return f"Error: '{path}' not found."
    return target.read_text(encoding="utf-8")


def list_files(directory: str = ".") -> str:
    """List files in a directory."""
    target = Path(directory)
    if not target.is_dir():
        return f"Error: '{directory}' is not a directory."
    entries = sorted(target.iterdir())
    return "\n".join(p.name + ("/" if p.is_dir() else "") for p in entries) or "(empty)"


def write_file(path: str, content: str) -> str:
    """Write content to a file."""
    target = Path(path)
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(content, encoding="utf-8")
    return f"Written to '{path}'."


def execute_python(code: str) -> str:
    """Execute Python code and return output."""
    try:
        result = subprocess.run(
            [sys.executable, "-c", code],
            capture_output=True,
            text=True,
            timeout=30,
        )
    except subprocess.TimeoutExpired:
        return "Error: Code execution timed out (30s limit)."

    if result.returncode != 0:
        return (
            f"Exit code {result.returncode}\n"
            f"STDOUT:\n{result.stdout}\n"
            f"STDERR:\n{result.stderr}"
        ).strip()
    return result.stdout.strip() or "(no output)"

각 함수가 `return`으로 돌려준 값은 `TinyAgent`가 받는 관찰 결과(`OBSERVATION`)입니다.

이제 이 함수들을 `NativeTools`에 등록합니다.

## 3. 파일 쓰기와 코드 실행 전 승인

파일 쓰기와 코드 실행은 작업 환경을 변경할 수 있습니다. 가령 모델이 자신의 설정 파일을 수정하면 프로그램이 더 이상 작동하지 않을 수 있습니다. 이번 예제에서는 `Tools`와 `NativeTools`를 수정해 지정한 도구를 실행하기 전에 사용자의 승인을 받습니다.

도구를 실행할 때 승인 대상인지 확인하고, 대상이면 사용자가 `y` 또는 `yes`를 입력할 때까지 실행을 보류합니다.

아래 코드에서 승인 여부를 검사하는 부분을 확인합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import execute_tool_annotated; execute_tool_annotated

`NativeTools`는 `Tools`를 상속하므로 같은 승인 절차를 사용합니다. `requires_approval`로 승인받아야 하는 함수를 지정합니다.

파일 쓰기와 코드 실행 도구에 승인이 필요하도록 등록합니다.

In [ ]:
from illustrated_agents.chapters.ch5 import NativeTools

# 일부 도구는 실행 전 승인 필요
tools = NativeTools(requires_approval=["write_file", "execute_python"])
tools.add_tool("read_file", read_file)
tools.add_tool("list_files", list_files)
tools.add_tool("write_file", write_file)
tools.add_tool("execute_python", execute_python)

`requires_approval`로 지정한 도구는 에이전트가 호출할 때마다 사용자의 확인을 받습니다.

파이썬 코드가 필요한 작업을 요청하고 실행 전에 확인 질문이 나오는지 살펴봅니다.

In [ ]:
from illustrated_agents.chapters.ch2 import Response

# 도구 호출 정의
response = Response(
    content="Some content...",
    reasoning="Some reasoning...",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)

# 도구 실행
tools.execute(response)

## 4. 터미널에 실행 과정 표시

모델의 추론, 행동, 도구 결과를 구분해 볼 수 있도록 터미널 출력을 구성합니다. [Rich](https://github.com/Textualize/rich)로 구현하는 `Display` 클래스는 생각 중인 상태와 답변 등을 화면에 표시합니다.

In [ ]:
BOLD = "\033[1m"
RESET = "\033[0m"
GREEN = "\033[32m"   # THOUGHT: 생각
RED = "\033[31m"     # ACTION: 행동
YELLOW = "\033[33m"  # OBSERVATION: 관찰
PURPLE = "\033[35m"  # ANSWER: 답변


class Display:
    """서식이 적용된 대화 화면이다."""

    def __call__(self, event: str, data: str | Response = None) -> None:

        # 응답 생성 중 상태 표시
        if event == "thinking":
            print(f"  Thinking...\n{RESET}")

        # THOUGHT: 생각
        elif event == "response":
            print(f"{BOLD}{GREEN}{'▒▒ THOUGHT ▒▒':<13}{RESET}")
            print(f"{data.reasoning}{RESET}\n")

            # ANSWER: 답변
            if data.content:
                print(f"{BOLD}{PURPLE}{'▒▒ ANSWER ▒▒':<13}{RESET}")
                print(f"{data.content}{RESET}\n")

        # ACTION: 행동
        elif event == "tool_call" and data:
            tool = data.tool_call["tool"]
            kwargs = data.tool_call["kwargs"]
            print(f"{BOLD}{RED}{'▒▒ ACTION ▒▒':<13}{RESET}")
            print(f"{tool}({kwargs}){RESET}\n")

        # OBSERVATION: 관찰
        elif event == "observation":
            print(f"{BOLD}{YELLOW}{'▒▒ OBSERVATION ▒▒':<13}{RESET}")
            print(f"{data}{RESET}\n")
            print(f"{'─' * 40}STEP{'─' * 40}{RESET}\n")

각 메서드가 어떤 값을 어떤 형식으로 표시하는지 살펴봅니다.

In [ ]:
from illustrated_agents.chapters.ch10 import display_annotated; display_annotated

`THOUGHT`, `ACTION`, `TOOL_CALL`, `OBSERVATION`에 해당하는 예시 값을 전달해 실제 에이전트를 실행하기 전에 출력 형태를 확인합니다.

In [ ]:
# 최종 답변 없이 도구 호출만 들어 있는 Response 생성
response = Response(
    content="I executed Python.",
    reasoning="Let's execute some python!",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)


# 응답 표시
display = Display()
display("thinking", response)
display("response", response)
display("tool_call", response)
display("observation", "Hello World!")

In [ ]:
# 최종 답변 없이 도구 호출만 들어 있는 Response 생성
response = Response(
    content="",
    reasoning="Let's execute some python!",
    tool_call= {
        "tool": "execute_python",
        "kwargs": {"code": "print('Hello World!')"}
    }
)


# 응답 표시
display = Display()
display("response", response)
display("tool_call", response)
display("observation", "Hello World!")

최종 답변이 어떻게 표시되는지도 확인합니다.

In [ ]:
# 도구 호출 없이 최종 답변만 들어 있는 Response 생성
response = Response(
    content="I believe that one plus one is two!",
    reasoning="I should answer the question.",
)

# 응답 표시
display = Display()
display("thinking", response)
display("response", response)

## 5. `TinyAgent`에 출력 연결

`TinyAgent`에 `Display`를 전달합니다. 모델이 답을 생성하거나 도구를 호출하고 관찰 결과를 받는 지점에 `self.display(...)`를 추가하면 각 단계의 진행 상황을 볼 수 있습니다.

In [ ]:
from illustrated_agents.chapters.ch10 import tinyagents_diff; tinyagents_diff

변경 사항을 반영한 전체 `TinyAgent` 클래스입니다.

In [ ]:
from illustrated_agents.chapters.ch2 import Response, Trajectory
from illustrated_agents.chapters.ch4 import Memory
from illustrated_agents.chapters.ch5 import Tools
from illustrated_agents.chapters.ch6 import ReAct


class TinyAgent:
    """교육용 에이전트의 기본 클래스를 정의한다."""

    def __init__(
        self,
        llm: LLM,
        memory: Memory,
        tools: Tools,
        planner: ReAct,
        display: Display,
    ):
        self.llm = llm
        self.memory = memory
        self.tools = tools
        self.planner = planner
        self.display = display

        self.trajectory = Trajectory()

        # 시스템 프롬프트에 각 구성 요소의 지침을 추가
        system_prompt = "You are a helpful assistant.\n\n"
        system_prompt += self.planner.prompt
        system_prompt += self.tools.prompt
        self.memory.add("system", system_prompt)

    def run(self, task: str, image_data: str = None) -> str:
        """주어진 작업으로 에이전트를 실행한다."""
        self.memory.add("user", task, image_data=image_data)
        self.trajectory.initialize(task)

        # 여러 단계를 수행하는 반복문
        for step in range(self.planner.max_steps):
            result = self._step()
            if result is not None:
                return result

        return "Max steps reached without completion."

    def _step(self) -> str | None:
        """에이전트의 한 단계를 실행한다."""
        # THOUGHT: 응답을 생성해 메모리에 추가
        self.display("thinking")
        response = self.llm.generate(
            self.memory.get_messages(), tools=self.tools.schemas
        )
        self.memory.add(
            "assistant", response.content, tool_call=response.tool_call
        )
        self.display("response", response)

        # 도구 호출 해석
        response = self.planner.parse(response)
        response = self.tools.parse(response)

        # ANSWER: 종료 조건 확인
        if self.tools.is_done(response):
            self.trajectory.add(response)
            return response.content

        return self._execute_action(response)

    def _execute_action(self, response: Response) -> None:
        """도구가 수행할 동작을 실행한다."""

        # ACTION: 도구 실행
        self.display("tool_call", response)
        result = self.tools.execute(response)

        # OBSERVATION: 도구 결과를 메모리에 추가하고 화면에 표시
        role, observation = self.tools.observation(result)
        self.memory.add(role, observation)
        self.trajectory.add(response, observation)
        self.display("observation", observation)

        return None

## 6. 구성 요소 연결과 실행

In [ ]:
from illustrated_agents.chapters.ch6 import NativeReAct

# 코딩 에이전트
agent = TinyAgent(
    llm=llm, 
    tools=tools, 
    memory=Memory(),
    planner=NativeReAct(),
    display=display
)

예를 들어 “How large is each file in the current folder?”(현재 폴더에 있는 각 파일의 크기는 얼마인가요?)라고 물어봅니다. 에이전트가 파일 크기를 조사하는 파이썬 코드를 작성하고 실행을 요청하는지 확인합니다.

In [ ]:
NAME = """
██████ ██ ███  ██ ██  ██   ▄████▄  ▄████  ██████ ███  ██ ██████
  ██   ██ ██ ▀▄██  ▀██▀    ██▄▄██ ██  ▄▄▄ ██▄▄   ██ ▀▄██   ██
  ██   ██ ██   ██   ██     ██  ██  ▀███▀  ██▄▄▄▄ ██   ██   ██
"""  # 간단한 ANSI 출력

def main():
  import shutil

  print(f"\n{NAME}\n")

  while True:
      try:
          query = input().strip()
          print("\033[0m")
      except (KeyboardInterrupt, EOFError):
          print("\033[0m", end="")
          break
      if not query or query.lower() in ("exit", "quit"):
          break
      try:
          agent.run(query)
      except Exception as e:
          print(f"ERROR: {e}\n")


if __name__ == "__main__":
  main()

에이전트가 선택한 도구와 코드 실행 승인 요청을 확인했습니다. 아래에서 대화 메시지도 확인합니다.

# 구현한 내용

파일을 읽고 쓰는 도구, 파이썬 코드를 표시하고 실행하는 도구, 사용자 승인 절차와 터미널 출력을 조합해 코딩 에이전트를 만들었습니다.

In [ ]:
from illustrated_agents.chapters.ch10 import what_we_built; what_we_built